Connected to .venv01 (3.13.11) (Python 3.13.11)

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from pathlib import Path
from arch.utility.exceptions import ConvergenceWarning


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

sys.modules.pop("srcB", None)

In [ ]:
FIT_YEAR = 2022
TEST_YEAR = 2023
YEARS = [FIT_YEAR, TEST_YEAR]
QUARTERS = ["q1", "q2", "q3", "q4"]

In [ ]:
# =============================================================================================================
# ======================================= LOAD OOS OUTPUTS ====================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/OOS/outputs_by_year/outputs_by_year_03.pkl", "rb") as f:
    outputs_by_year = pickle.load(f)


# =============================================================================================================
# ======================================= IMPORTS FROM SRC ====================================================
# =============================================================================================================

from srcB.market import *
from srcB.helpers import *


# =============================================================================================================
# ======================================= READ D_BTC AND DF ===================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")


# =============================================================================================================
# ======================================= SET INDEX TO DATETIME ===============================================
# =============================================================================================================

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(
        pd.to_datetime(d_btc_raw_by_year[y]["date_time"])
    )

    df_raw_by_year[y] = df_raw_by_year[y].set_index(
        pd.to_datetime(df_raw_by_year[y]["date_time"])
    )


# =============================================================================================================
# ======================================= LOAD X_BY_YEAR ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/OOS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)


for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2


Xq_by_year = {}

for y in YEARS:

    Xq_by_year[y] = {}

    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]

    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]

    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]

    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]


# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)


for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[
            :, ~Xq_by_year[y][q].columns.duplicated(keep="first")
        ]


# =============================================================================================================
# ======================================= BORROW RFR COLUMN ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/OOS/ready_data/d_btc_by_year_all.pkl", "rb") as f:
    d_btc_by_year = pickle.load(f)


for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(
        d_btc_raw_by_year[y].index,
        method="ffill"
    )

In [ ]:
# =============================================================================================================
# ======================================= TCN_2022_FIT =========================================================
# =============================================================================================================

sys.modules.pop("srcB.tcn_model_v2", None)
from srcB.tcn_model_v2 import *

X_fit = pd.concat(
    [Xq_by_year[FIT_YEAR][q] for q in QUARTERS]
).sort_index().copy()

garch_fit = pd.concat(
    [outputs_by_year[FIT_YEAR][q]["garch"] for q in QUARTERS]
).sort_index()

X_fit["garch"] = garch_fit.reindex(X_fit.index)


tcn_fit_output, tcn_bundle, tcn_targets = tcn_mfiv(
    X_df=X_fit,
    option_df=df_raw_by_year[FIT_YEAR],
    time_df=d_btc_raw_by_year[FIT_YEAR],
    sequence_length=168,
    forward_hours=24,
    delta_target=0.20
)


print("fit year:", FIT_YEAR)
print("fit observations:", len(tcn_fit_output))

fit year: 2022
fit observations: 8521


In [ ]:
# =============================================================================================================
# ======================================= PREPARE 2023 PREDICTION INPUT =======================================
# =============================================================================================================

PREDICTION_YEARS = [FIT_YEAR, TEST_YEAR]

X_test = pd.concat([
    Xq_by_year[y][q]
    for y in PREDICTION_YEARS
    for q in QUARTERS
]).sort_index().copy()

garch_test = pd.concat([
    outputs_by_year[y][q]["garch"]
    for y in PREDICTION_YEARS
    for q in QUARTERS
]).sort_index()

X_test["garch"] = garch_test.reindex(X_test.index)

In [ ]:
# =============================================================================================================
# ======================================= TCN_2023_TEST_PREDICTION ============================================
# =============================================================================================================

tcn_test_output = predict_tcn(
    tcn_bundle,
    X_test
)

tcn_test_output = tcn_test_output[
    tcn_test_output.index.year == TEST_YEAR
].copy()


print(tcn_test_output[["tcn_action", "tcn_best_score"]].head())
print(tcn_test_output["tcn_action"].value_counts())

                               tcn_action  tcn_best_score
2023-01-08 09:00:00+00:00  short_strangle        0.000889
2023-01-08 10:00:00+00:00  short_strangle        0.001015
2023-01-08 11:00:00+00:00  short_straddle       11.974154
2023-01-08 12:00:00+00:00  short_strangle        9.917077
2023-01-08 13:00:00+00:00  short_straddle        4.235417
tcn_action
long_straddle     3126
short_straddle    2642
no_trade          1435
short_strangle    1116
long_strangle      264
Name: count, dtype: int64


In [ ]:
# =============================================================================================================
# ======================================= ADD TCN TEST OUTPUT ==================================================
# =============================================================================================================

for q in QUARTERS:
    q_index = Xq_by_year[TEST_YEAR][q].index
    tcn_q = tcn_test_output.reindex(q_index)

    outputs_by_year[TEST_YEAR][q] = outputs_by_year[TEST_YEAR][q].drop(
        columns=tcn_q.columns,
        errors="ignore"
    )

    outputs_by_year[TEST_YEAR][q] = pd.concat(
        [outputs_by_year[TEST_YEAR][q], tcn_q],
        axis=1
    )

In [ ]:
# =============================================================================================================
# ======================================= PREPARE TEST TRADER INPUT ============================================
# =============================================================================================================

for q in QUARTERS:

    outputs_by_year[TEST_YEAR][q]["rv"] = Xq_by_year[TEST_YEAR][q]["rv"].reindex(
        outputs_by_year[TEST_YEAR][q].index
    )

    outputs_by_year[TEST_YEAR][q]["MFIV_24"] = Xq_by_year[TEST_YEAR][q]["MFIV_24"].reindex(
        outputs_by_year[TEST_YEAR][q].index
    )


trader_cache = {
    TEST_YEAR: pd.concat(
        [outputs_by_year[TEST_YEAR][q] for q in QUARTERS]
    ).sort_index()
}

In [ ]:
# =============================================================================================================
# ======================================= TCN_2023_TEST ========================================================
# =============================================================================================================

from srcB.market import Market

sys.modules.pop("srcB.trader_testing_tcn", None)
from srcB.trader_testing_tcn import testing_execute_trading_strategy


market = Market(
    option_df=df_raw_by_year[TEST_YEAR],
    time_df=d_btc_raw_by_year[TEST_YEAR],
    starting_cash=100_000,
    im_multiplier=0.15,
    mm_buffer_rate=0.05,
)


snapshots, greek_tests = testing_execute_trading_strategy(
    bt_year=TEST_YEAR,
    bt_quarter=None,
    model="tcn_mfiv",
    outputs_by_year=outputs_by_year,
    market=market,
    trader_cache=trader_cache,
    run_by="year",
    tcn_score_threshold=0.003
)


equity = pd.Series(
    [s["equity"] for s in snapshots],
    index=[s["time"] for s in snapshots]
)


returns = equity.pct_change().dropna()

sharpe = (
    returns.mean()
    / returns.std()
    * np.sqrt(24 * 365)
)


print("year:", TEST_YEAR)
print("trades:", len(greek_tests))
print("sharpe:", sharpe)
print("final equity:", equity.iloc[-1])

year: 2023
trades: 174
sharpe: 0.11236980054012081
final equity: 100165.83259749993


In [ ]:
# =============================================================================================================
# ======================================= EXCESS RETURN SHARPE =================================================
# =============================================================================================================

rfr_annual = 0.045

rfr_hourly = (
    (1 + rfr_annual) ** (1 / (24 * 365))
    - 1
)

excess_returns = returns - rfr_hourly

sharpe_excess = (
    excess_returns.mean()
    / excess_returns.std()
    * np.sqrt(24 * 365)
)

print("excess sharpe:", sharpe_excess)

excess sharpe: -0.09829302523749282
